# Visualisation des villes d'Australie

In [1]:
pip install plotly pandas requests

In [5]:
import pandas as pd
import plotly.graph_objects as go
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import os

In [6]:
pred=pd.read_csv("/content/sample_data/predict_2017-01-19.csv")
pred.head(10)


,Ville,Date_Predite,Temp_Predite,Prob_Pluie,Pluie_Prevue
0,Adelaide,2017-01-19,32.54,0.9783,1
1,Albany,2017-01-19,22.49,0.0244,0
2,Albury,2017-01-19,31.36,0.9615,1
3,AliceSprings,2017-01-19,35.68,0.0605,0
4,BadgerysCreek,2017-01-19,31.83,0.9288,1
5,Ballarat,2017-01-19,26.54,0.9495,1
6,Bendigo,2017-01-19,28.14,0.6573,1
7,Brisbane,2017-01-19,33.47,0.0040,0
8,Cairns,2017-01-19,30.60,0.6492,1
9,Canberra,2017-01-19,33.95,0.6663,1


In [7]:
pred.columns=pred.columns.str.lower()

In [8]:
pred.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ville         49 non-null     object 
 1   date_predite  49 non-null     object 
 2   temp_predite  49 non-null     float64
 3   prob_pluie    49 non-null     float64
 4   pluie_prevue  49 non-null     int64  
dtypes: float64(2), int64(1), object(2)
memory usage: 2.0+ KB


In [7]:
#On charge les données historiques de météo
historical_weather=pd.read_csv("/content/sample_data/clean_data.csv")
historical_weather.head(10)


,Date,Location,MinTemp,MaxTemp,Rainfall,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,WindSpeed9am,...,RainTomorrow,Month,TempRange,Humidity_Avg,Pressure_Diff,WindDir3pm_sin,WindDir3pm_cos,City_Encoded,Month_sin,Month_cos
0,2008-12-01,Albury,13.4,22.9,0.6,W,44.0,W,WNW,20.0,...,0.0,12.0,9.5,46.5,-0.6,-0.923880,3.826834e-01,2.0,-2.449294e-16,1.0
1,2008-12-02,Albury,7.4,25.1,0.0,WNW,44.0,NNW,WSW,4.0,...,0.0,12.0,17.7,34.5,-2.8,-0.923880,-3.826834e-01,2.0,-2.449294e-16,1.0
2,2008-12-03,Albury,12.9,25.7,0.0,WSW,46.0,W,WSW,19.0,...,0.0,12.0,12.8,34.0,1.1,-0.923880,-3.826834e-01,2.0,-2.449294e-16,1.0
3,2008-12-04,Albury,9.2,28.0,0.0,NE,24.0,SE,E,11.0,...,0.0,12.0,18.8,30.5,-4.8,1.000000,6.123234e-17,2.0,-2.449294e-16,1.0
4,2008-12-05,Albury,17.5,32.3,1.0,W,41.0,ENE,NW,7.0,...,0.0,12.0,14.8,57.5,-4.8,-0.707107,7.071068e-01,2.0,-2.449294e-16,1.0
5,2008-12-06,Albury,14.6,29.7,0.2,WNW,56.0,W,W,19.0,...,0.0,12.0,15.1,39.0,-3.8,-1.000000,-1.836970e-16,2.0,-2.449294e-16,1.0
6,2008-12-07,Albury,14.3,25.0,0.0,W,50.0,SW,W,20.0,...,0.0,12.0,10.7,34.0,-1.4,-1.000000,-1.836970e-16,2.0,-2.449294e-16,1.0
7,2008-12-08,Albury,7.7,26.7,0.0,W,35.0,SSE,W,6.0,...,0.0,12.0,19.0,33.5,-3.3,-1.000000,-1.836970e-16,2.0,-2.449294e-16,1.0
8,2008-12-09,Albury,9.7,31.9,0.0,NNW,80.0,SE,NW,7.0,...,1.0,12.0,22.2,25.5,-5.3,-0.707107,7.071068e-01,2.0,-2.449294e-16,1.0
9,2008-12-10,Albury,13.1,30.1,1.4,W,28.0,S,SSE,15.0,...,0.0,12.0,17.0,42.5,-1.3,0.382683,-9.238795e-01,2.0,-2.449294e-16,1.0


In [9]:
historical_weather.columns

Index(['Date', 'Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'WindGustDir',
       'WindGustSpeed', 'WindDir9am', 'WindDir3pm', 'WindSpeed9am',
       'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am',
       'Pressure3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'RainTomorrow',
       'Month', 'TempRange', 'Humidity_Avg', 'Pressure_Diff', 'WindDir3pm_sin',
       'WindDir3pm_cos', 'City_Encoded', 'Month_sin', 'Month_cos'],
      dtype='object')

## Harmonisation des colonnes : petit data ingéneriing

In [18]:
import pandas as pd
import plotly.graph_objects as go
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import os

# ── Chargement ──────────────────────────────────────────────────────
hist = pd.read_csv("/content/sample_data/clean_data.csv")
pred = pd.read_csv("/content/sample_data/predict_2017-01-19.csv")

pred.columns=pred.columns.str.lower()


# Harmonisation des noms de colonnes
hist = hist.rename(columns={"Location": "city"})
pred = pred.rename(columns={"ville": "city", "date_predite": "date"})

# ── Géocodage avec cache ────────────────────────────────────────────
CACHE_FILE = "city_coords_cache.csv"

if os.path.exists(CACHE_FILE):
    coords_df = pd.read_csv(CACHE_FILE)
else:
    geolocator = Nominatim(user_agent="weather_map")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    cities = hist["city"].unique()
    cache = []
    for city in cities:
        loc = geocode(f"{city}, Australia")
        if loc:
            cache.append({"city": city, "lat": loc.latitude, "lon": loc.longitude})
    coords_df = pd.DataFrame(cache)
    coords_df.to_csv(CACHE_FILE, index=False)

# ── Agrégation historique par ville ────────────────────────────────
avg = hist.groupby("city").agg(
    rainfall_avg   = ("Rainfall",     "mean"),
    temp_max_avg   = ("MaxTemp",      "mean"),
    humidity_avg   = ("Humidity_Avg", "mean"),
    rain_days_pct  = ("RainToday",    lambda x: (x == "Yes").mean() * 100),  # % jours de pluie
).reset_index()

# Fusion coords
avg = avg.merge(coords_df, on="city", how="left")

# ── Zone climatique déduite des précipitations ──────────────────────
def assign_zone(row):
    if row["rainfall_avg"] > 5:
        return "Tropical"
    elif row["rainfall_avg"] > 2:
        return "Temperate"
    elif row["humidity_avg"] > 60:
        return "Mediterranean"
    else:
        return "Arid"

avg["climate_zone"] = avg.apply(assign_zone, axis=1)

zone_colors = {
    "Tropical":      "#2d6a4f",
    "Temperate":     "#457b9d",
    "Mediterranean": "#9b5de5",
    "Arid":          "#e76f51",
}

# ── Prédictions : 1 ligne par ville par jour (J+1, J+2, J+3) ───────
pred["day_offset"] = pd.to_datetime(pred["date"]).rank(method="dense").astype(int)
pred = pred.merge(coords_df, on="city", how="left")

# ── Construction de la carte ────────────────────────────────────────
fig = go.Figure()

# Bulles historiques colorées par zone climatique
for zone, color in zone_colors.items():
    subset = avg[avg["climate_zone"] == zone]
    fig.add_trace(go.Scattergeo(
        lat=subset["lat"],
        lon=subset["lon"],
        mode="markers",
        marker=dict(
            size=subset["rainfall_avg"] * 4 + 12,
            color=color,
            opacity=0.75,
            line=dict(width=1.5, color="white"),
            sizemode="area",
        ),
        text=subset.apply(lambda r: (
            f"<b>{r['city']}</b><br>"
            f"Zone : {r['climate_zone']}<br>"
            f"Pluie moy : {r['rainfall_avg']:.1f} mm<br>"
            f"Temp max moy : {r['temp_max_avg']:.1f} °C<br>"
            f"Humidité moy : {r['humidity_avg']:.1f} %<br>"
            f"Jours de pluie : {r['rain_days_pct']:.0f} %"
        ), axis=1),
        hoverinfo="text",
        name=zone,
    ))

# Marqueurs prédictions J+1 / J+2 / J+3
day_labels = {1: "J+1", 2: "J+2", 3: "J+3"}
for day in [1, 2, 3]:
    day_pred = pred[pred["day_offset"] == day]
    fig.add_trace(go.Scattergeo(
        lat=day_pred["lat"],
        lon=day_pred["lon"],
        mode="markers+text",
        marker=dict(
            size=12,
            color=day_pred["prob_pluie"].apply(lambda p: f"rgba(0, 120, 200, {p:.2f})"),
            symbol="diamond",
            line=dict(width=2, color="white"),
        ),
        text=day_pred["prob_pluie"].apply(lambda p: f"{p*100:.0f}%"),
        textposition="top center",
        hovertext=day_pred.apply(lambda r: (
            f"<b>{r['city']}</b> — {day_labels[day]}<br>"
            f"Temp prédite : {r['temp_predite']:.1f} °C<br>"
            f"Prob. pluie : {r['prob_pluie']*100:.0f} %<br>"
            f"Pluie prévue : {'🌧️ Oui' if r['pluie_prevue'] else '☀️ Non'}"
        ), axis=1),
        hoverinfo="text",
        name=f"Prédiction {day_labels[day]}",
        visible=(day == 1),
    ))

# ── Mise en page ────────────────────────────────────────────────────
n_zones = len(zone_colors)

fig.update_layout(
    title=dict(text="🌧️ Météo Australie — Historique & Prédictions", font_size=20),
    geo=dict(
        scope="world",
        center=dict(lat=-25, lon=134),
        projection_scale=3.5,
        showland=True,    landcolor="#f8f4e3",
        showocean=True,   oceancolor="#caf0f8",
        showlakes=True,   lakecolor="#90e0ef",
        showcoastlines=True, coastlinecolor="#444",
        showcountries=True,  countrycolor="#aaa",
        showframe=False,
    ),
    legend=dict(title="Zones & prédictions", bgcolor="rgba(255,255,255,0.85)"),
    paper_bgcolor="#f0f0f0",
    height=750,
    updatemenus=[dict(
        type="buttons",
        direction="right",
        x=0.5, y=1.05,
        buttons=[
            dict(label="J+1", method="update", args=[{"visible": [True]*n_zones + [True,  False, False]}]),
            dict(label="J+2", method="update", args=[{"visible": [True]*n_zones + [False, True,  False]}]),
            dict(label="J+3", method="update", args=[{"visible": [True]*n_zones + [False, False, True ]}]),
        ]
    )]
)

fig.write_html("australia_weather_map.html")
fig.show()

### Partie clean

In [10]:
#coordonnées géographiques
CACHE_FILE = "city_coords_cache.csv"

if os.path.exists(CACHE_FILE):
    coords_df = pd.read_csv(CACHE_FILE)
else:
    geolocator = Nominatim(user_agent="weather_map")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    cache = []
    for city in pred["ville"].unique():
        loc = geocode(f"{city}, Australia")
        if loc:
            cache.append({"ville": city, "lat": loc.latitude, "lon": loc.longitude})
    coords_df = pd.DataFrame(cache)
    coords_df.to_csv(CACHE_FILE, index=False)

In [11]:
coords_df

,city,lat,lon
0,Albury,-36.073773,146.913526
1,Cobar,-31.966663,145.304505
2,Moree,-29.461720,149.840715
3,Newcastle,-32.919295,151.779535
4,Penrith,-33.751195,150.694171
5,Richmond,-37.807450,144.990721
6,Sydney,-33.869844,151.208285
7,Williamtown,-32.815000,151.842778
8,Wollongong,-34.424394,150.893850
9,Canberra,-35.297591,149.101268


#Fusion des 2 datasets

In [12]:
coords_df = coords_df.rename(columns={"city": "ville"})


In [13]:
pred = pred.merge(coords_df, on="ville", how="left")

In [14]:
pred

,ville,date_predite,temp_predite,prob_pluie,pluie_prevue,lat,lon
0,Adelaide,2017-01-19,32.54,0.9783,1,-34.928181,138.599931
1,Albany,2017-01-19,22.49,0.0244,0,-35.024782,117.883608
2,Albury,2017-01-19,31.36,0.9615,1,-36.073773,146.913526
3,AliceSprings,2017-01-19,35.68,0.0605,0,NaN,NaN
4,BadgerysCreek,2017-01-19,31.83,0.9288,1,NaN,NaN
5,Ballarat,2017-01-19,26.54,0.9495,1,-37.562301,143.860565
6,Bendigo,2017-01-19,28.14,0.6573,1,-36.759018,144.282672
7,Brisbane,2017-01-19,33.47,0.0040,0,-27.468962,153.023501
8,Cairns,2017-01-19,30.60,0.6492,1,-16.920666,145.772185
9,Canberra,2017-01-19,33.95,0.6663,1,-35.297591,149.101268


In [15]:
pred["date_predite"] = pd.to_datetime(pred["date_predite"])


In [17]:
# ── Dates uniques triées ─────────────────────────────────────────────
dates = sorted(pred["date_predite"].unique())

# ── Couleurs par probabilité de pluie ──────────────────────────────
def rain_color(prob):
    """Du beige sec au bleu orage selon la prob de pluie."""
    if prob < 0.2:   return "#F4A261"   # sec - orange
    elif prob < 0.4: return "#90C4E4"   # légère - bleu clair
    elif prob < 0.6: return "#4A90D9"   # modéré - bleu
    elif prob < 0.8: return "#1F5FA6"   # élevée - bleu foncé
    else:            return "#0A2E66"   # très élevée - bleu nuit

# ── Icône météo dans le tooltip ─────────────────────────────────────
def weather_icon(row):
    if row["pluie_prevue"] == 1:
        return "🌧️ Pluie prévue"
    elif row["prob_pluie"] > 0.4:
        return "⛅ Risque modéré"
    else:
        return "☀️ Temps sec"


In [19]:

# ── Icône météo dans le tooltip ─────────────────────────────────────
def weather_icon(row):
    if row["pluie_prevue"] == 1:
        return "🌧️ Pluie prévue"
    elif row["prob_pluie"] > 0.4:
        return "⛅ Risque modéré"
    else:
        return "☀️ Temps sec"

# ── Construction de la figure ────────────────────────────────────────
fig = go.Figure()

for date in dates:
    df_day = pred[pred["date_predite"] == date].copy()
    df_day["color"]   = df_day["prob_pluie"].apply(rain_color)
    df_day["icon"]    = df_day.apply(weather_icon, axis=1)
    df_day["bubble"]  = df_day["prob_pluie"] * 35 + 12

    fig.add_trace(go.Scattergeo(
        lat=df_day["lat"],
        lon=df_day["lon"],
        visible=False,
        mode="markers+text",
        marker=dict(
            size=df_day["bubble"],
            color=df_day["color"],
            opacity=0.88,
            line=dict(width=1.5, color="rgba(255,255,255,0.4)"),
        ),
        text=df_day["ville"],
        textfont=dict(size=11, color="white", family="monospace"),
        textposition="top center",
        hovertemplate=(
            "<b style='font-size:14px'>%{customdata[0]}</b><br>"
            "<span style='color:#90C4E4'>%{customdata[3]}</span><br>"
            "─────────────────<br>"
            "🌡️ Température  : <b>%{customdata[1]:.1f} °C</b><br>"
            "🌧️ Prob. pluie  : <b>%{customdata[2]:.0f} %</b><br>"
            "<extra></extra>"
        ),
        customdata=df_day[["ville", "temp_predite", "prob_pluie_pct", "icon"]].assign(
            prob_pluie_pct=df_day["prob_pluie"] * 100
        ).values if False else list(zip(
            df_day["ville"],
            df_day["temp_predite"],
            df_day["prob_pluie"] * 100,
            df_day["icon"],
        )),
        name=str(date.date()),
    ))

# Rendre le premier trace visible
fig.data[0].visible = True

# ── Boutons dates dynamiques ────────────────────────────────────────
n = len(dates)
buttons = []

for i, date in enumerate(dates):
    label = pd.Timestamp(date).strftime("%-d %b %Y")   # ex: "3 Jan 2025"
    visibility = [j == i for j in range(n)]
    buttons.append(dict(
        label=label,
        method="update",
        args=[{"visible": visibility},
              {"title": f"<b>🌏 Météo Australie</b>  ·  {label}"}],
    ))

# ── Mise en page premium ────────────────────────────────────────────
first_label = pd.Timestamp(dates[0]).strftime("%-d %b %Y")

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0b1b2e",
    plot_bgcolor="#0b1b2e",
    font=dict(family="monospace", color="#d0dce8"),
    title=dict(
        text=f"<b>🌏 Météo Australie</b>  ·  {first_label}",
        font=dict(size=22, color="#e8f1fa"),
        x=0.5,
        y=0.97,
    ),
    geo=dict(
        scope="world",
        center=dict(lat=-25, lon=134),
        projection_scale=3.8,
        bgcolor="#0b1b2e",
        showland=True,       landcolor="#1c2f45",
        showocean=True,      oceancolor="#091525",
        showlakes=True,      lakecolor="#122136",
        showcoastlines=True, coastlinecolor="#2e4a66",  coastlinewidth=1.2,
        showcountries=True,  countrycolor="#1e3550",
        showframe=False,
        showrivers=True,     rivercolor="#122136",
        showsubunits=True,   subunitcolor="#1a2e45",
    ),
    height=720,
    margin=dict(l=0, r=0, t=60, b=60),

    # ── Barre de boutons dates ──────────────────────────────────────
    updatemenus=[dict(
        type="buttons",
        direction="right",
        x=0.5, xanchor="center",
        y=-0.05, yanchor="top",
        bgcolor="#112236",
        bordercolor="#2e4a66",
        borderwidth=1,
        font=dict(size=13, color="#90C4E4", family="monospace"),
        buttons=buttons,
        pad=dict(r=8, t=8, b=8),
    )],

    # ── Légende couleurs ────────────────────────────────────────────
    annotations=[
        dict(x=0.01, y=0.28, xref="paper", yref="paper", showarrow=False,
             align="left", bgcolor="#0d1e30", bordercolor="#2e4a66", borderwidth=1,
             borderpad=8,
             text=(
                 "<b style='color:#d0dce8'>Probabilité de pluie</b><br>"
                 "<span style='color:#F4A261'>●</span> &lt; 20 %  — Sec<br>"
                 "<span style='color:#90C4E4'>●</span> 20–40 %  — Légère<br>"
                 "<span style='color:#4A90D9'>●</span> 40–60 %  — Modérée<br>"
                 "<span style='color:#1F5FA6'>●</span> 60–80 %  — Élevée<br>"
                 "<span style='color:#0A2E66'>●</span> &gt; 80 %  — Très élevée"
             ),
             font=dict(size=12, family="monospace", color="#90C4E4")),
    ],
)

fig.write_html("australia_weather_map.html")
fig.show()
print("Carte Météorologique" )

Carte Météorologique


In [20]:

# ── Icône météo selon prob + pluie_prevue ───────────────────────────
def get_icon(prob, pluie):
    if pluie == 1 and prob >= 0.7: return "⛈"
    if pluie == 1 and prob >= 0.4: return "🌧"
    if pluie == 1:                 return "🌦"
    if prob >= 0.4:                return "⛅"
    if prob >= 0.15:               return "🌤"
    return "☀"

# ── Couleur badge température selon chaleur ──────────────────────────
def badge_color(temp):
    if temp >= 35:  return "#c0392b"   # rouge brûlant
    if temp >= 28:  return "#e67e22"   # orange chaud
    if temp >= 20:  return "#e8a020"   # ambre
    if temp >= 12:  return "#2980b9"   # bleu tempéré
    return "#1a5276"                   # bleu froid

# ── Labels jours en français ─────────────────────────────────────────
DAY_FR   = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
MONTH_FR = ["", "Jan", "Fév", "Mar", "Avr", "Mai", "Jun",
             "Jul", "Aoû", "Sep", "Oct", "Nov", "Déc"]

# ── Figure ───────────────────────────────────────────────────────────
fig      = go.Figure()
n_dates  = len(dates)

for i, date in enumerate(dates):
    df_day          = pred[pred["date_predite"] == date].copy()
    df_day["icon"]  = df_day.apply(lambda r: get_icon(r["prob_pluie"], r["pluie_prevue"]), axis=1)
    df_day["badge"] = df_day["temp_predite"].apply(badge_color)
    visible         = (i == 0)

    # 1) Icônes météo — gros emoji centré sur la ville
    fig.add_trace(go.Scattergeo(
        lat=df_day["lat"],
        lon=df_day["lon"],
        visible=visible,
        mode="text",
        text=df_day["icon"],
        textfont=dict(size=30, family="Arial"),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "%{customdata[3]}<br>"
            "─────────────<br>"
            "🌡 Temp : <b>%{customdata[1]:.0f} °C</b><br>"
            "💧 Pluie : <b>%{customdata[2]:.0f} %</b><br>"
            "<extra></extra>"
        ),
        customdata=list(zip(
            df_day["ville"],
            df_day["temp_predite"],
            df_day["prob_pluie"] * 100,
            df_day["icon"],
        )),
        showlegend=False,
        name=f"icon_{i}",
    ))

    # 2) Badge température — cercle coloré + texte blanc en dessous à droite
    fig.add_trace(go.Scattergeo(
        lat=df_day["lat"] - 1.8,
        lon=df_day["lon"] + 2.2,
        visible=visible,
        mode="markers+text",
        marker=dict(
            size=28,
            color=df_day["badge"],
            symbol="circle",
            line=dict(width=0),
            opacity=0.92,
        ),
        text=df_day["temp_predite"].apply(lambda t: f"<b>{t:.0f}°</b>"),
        textfont=dict(size=11, color="white", family="Arial Black"),
        textposition="middle center",
        hoverinfo="skip",
        showlegend=False,
        name=f"temp_{i}",
    ))

    # 3) Nom de la ville en petit sous l'icône
    fig.add_trace(go.Scattergeo(
        lat=df_day["lat"] + 2.5,
        lon=df_day["lon"],
        visible=visible,
        mode="text",
        text=df_day["ville"],
        textfont=dict(size=9.5, color="#1a3a5c", family="Arial"),
        hoverinfo="skip",
        showlegend=False,
        name=f"label_{i}",
    ))

# ── Boutons dates (style onglets) ────────────────────────────────────
buttons = []
for i, date in enumerate(dates):
    ts    = pd.Timestamp(date)
    label = f"<b>{DAY_FR[ts.dayofweek]}</b><br>{ts.day}<br>{MONTH_FR[ts.month].upper()}"
    # 3 traces par date
    vis   = []
    for j in range(n_dates):
        vis += [j == i, j == i, j == i]
    buttons.append(dict(label=label, method="update", args=[{"visible": vis}]))

# ── Mise en page ─────────────────────────────────────────────────────
fig.update_layout(
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Arial"),

    title=dict(
        text="<b>Météo Australie</b>",
        font=dict(size=21, color="#1a3a5c", family="Arial"),
        x=0.01, y=0.98, xanchor="left",
    ),

    geo=dict(
        scope="world",
        center=dict(lat=-25, lon=134),
        projection_scale=3.8,
        bgcolor="#b8d4e8",
        # Terre en bleu moyen — comme la chaîne météo
        showland=True,       landcolor="#5b9ec9",
        showocean=True,      oceancolor="#aacde0",
        showlakes=True,      lakecolor="#aacde0",
        showcoastlines=True, coastlinecolor="white", coastlinewidth=1.8,
        showcountries=True,  countrycolor="white",   countrywidth=1.8,
        showsubunits=True,   subunitcolor="white",   subunitwidth=1.2,
        showframe=False,
        showrivers=False,
        resolution=50,
    ),

    height=700,
    margin=dict(l=10, r=10, t=110, b=20),

    # ── Onglets dates en haut ─────────────────────────────────────
    updatemenus=[dict(
        type="buttons",
        direction="right",
        x=0.5,  xanchor="center",
        y=1.13, yanchor="bottom",
        active=0,
        bgcolor="white",
        bordercolor="#c0d4e4",
        borderwidth=1,
        font=dict(size=11, family="Arial", color="#1a3a5c"),
        buttons=buttons,
        pad=dict(r=3, l=3, t=6, b=6),
    )],

    # ── Légende compacte ──────────────────────────────────────────
    annotations=[dict(
        x=0.01, y=0.03,
        xref="paper", yref="paper",
        showarrow=False, align="left",
        bgcolor="white",
        bordercolor="#c0d4e4", borderwidth=1,
        borderpad=8,
        text=(
            "<span style='font-size:11px; color:#1a3a5c'>"
            "☀ Sec &nbsp; ⛅ Variable &nbsp; 🌧 Pluie &nbsp; ⛈ Orage"
            "</span>"
        ),
        font=dict(size=11, family="Arial"),
    )],
)

fig.write_html("australia_weather_map.html")
fig.show()
print("Carte exportée → australia_weather_map.html")

Carte exportée → australia_weather_map.html


# Test

In [22]:
pip install pydeck

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 121.1 MB/s eta 0:00:00


In [23]:

import pydeck as pdk

# Map of San Francisco from 1906
IMG_URL = "https://i.imgur.com/W95ked7.jpg"

# Specifies the corners of the image bounding box
BOUNDS = [
    [-122.52690000000051, 37.70313158980733],
    [-122.52690000000051, 37.816395657523195],
    [-122.34604834372873, 37.816134829424335],
    [-122.34656848822227, 37.70339041384273],
]

bitmap_layer = pdk.Layer("BitmapLayer", image=IMG_URL, bounds=BOUNDS, opacity=0.7)

view_state = pdk.ViewState(
    latitude=37.7576171,
    longitude=-122.5776844,
    zoom=10,
    bearing=-45,
    pitch=60,
)

r = pdk.Deck(bitmap_layer, initial_view_state=view_state, map_provider="mapbox", map_style=pdk.map_styles.SATELLITE)

r.to_html("bitmap_layer.html")

<IPython.core.display.Javascript object>

In [24]:
pip install folium geopandas

In [34]:
pred=pred.dropna()

In [44]:
pred

,ville,date_predite,temp_predite,prob_pluie,pluie_prevue,lat,lon
0,Adelaide,2017-01-19,32.54,0.9783,1,-34.928181,138.599931
1,Albany,2017-01-19,22.49,0.0244,0,-35.024782,117.883608
2,Albury,2017-01-19,31.36,0.9615,1,-36.073773,146.913526
5,Ballarat,2017-01-19,26.54,0.9495,1,-37.562301,143.860565
6,Bendigo,2017-01-19,28.14,0.6573,1,-36.759018,144.282672
7,Brisbane,2017-01-19,33.47,0.0040,0,-27.468962,153.023501
8,Cairns,2017-01-19,30.60,0.6492,1,-16.920666,145.772185
9,Canberra,2017-01-19,33.95,0.6663,1,-35.297591,149.101268
10,Cobar,2017-01-19,40.11,0.9562,1,-31.966663,145.304505
12,Dartmoor,2017-01-19,26.21,0.8903,1,-37.895212,141.267943


In [35]:
import pandas as pd
import geopandas as gpd
import folium
import json
import os
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter


# ── Icône météo HTML (vraie icône CSS, pas emoji) ───────────────────
def weather_icon_html(prob, pluie, temp):
    if pluie == 1 and prob >= 0.7:
        icon, color = "⛈", "#2c3e50"
    elif pluie == 1 and prob >= 0.4:
        icon, color = "🌧", "#2980b9"
    elif pluie == 1:
        icon, color = "🌦", "#5dade2"
    elif prob >= 0.4:
        icon, color = "⛅", "#85929e"
    elif prob >= 0.15:
        icon, color = "🌤", "#f39c12"
    else:
        icon, color = "☀", "#f1c40f"

    badge_color = "#c0392b" if temp >= 35 else \
                  "#e67e22" if temp >= 28 else \
                  "#e8a020" if temp >= 20 else "#2980b9"

    return f"""
    <div style="
        display: flex; flex-direction: column; align-items: center;
        filter: drop-shadow(0 2px 4px rgba(0,0,0,0.3));
        font-family: Arial, sans-serif;
    ">
        <div style="font-size: 36px; line-height: 1;">{icon}</div>
        <div style="
            background: {badge_color};
            color: white;
            font-size: 12px;
            font-weight: bold;
            padding: 2px 7px;
            border-radius: 4px;
            margin-top: 3px;
            border-right: 3px solid rgba(0,0,0,0.25);
        ">{temp:.0f}°</div>
    </div>"""

# ── Popup HTML détaillé ─────────────────────────────────────────────
def popup_html(row):
    icon = "🌧" if row["pluie_prevue"] == 1 else "☀"
    return f"""
    <div style="font-family:Arial,sans-serif; min-width:160px; padding:4px">
        <b style="font-size:15px">{row['ville']}</b><br>
        <span style="color:#888; font-size:11px">{pd.Timestamp(row['date_predite']).strftime('%d %b %Y')}</span>
        <hr style="margin:6px 0; border-color:#eee">
        <table style="font-size:13px; width:100%">
            <tr><td>🌡 Température</td><td><b>{row['temp_predite']:.1f} °C</b></td></tr>
            <tr><td>💧 Prob. pluie</td><td><b>{row['prob_pluie']*100:.0f} %</b></td></tr>
            <tr><td>🌂 Pluie prévue</td><td><b>{icon} {'Oui' if row['pluie_prevue']==1 else 'Non'}</b></td></tr>
        </table>
    </div>"""

# ── Couche pluie par état (choroplèthe) ─────────────────────────────
# Moyenne prob_pluie par état selon les villes qui y sont
state_rain = pred.groupby("ville")["prob_pluie"].mean().reset_index()

# ── Construction carte Folium ────────────────────────────────────────
m = folium.Map(
    location=[-25, 134],
    zoom_start=4,
    tiles="CartoDB Positron",   # fond propre et pro
    control_scale=True,
)

# Fond CartoDB alternatif : "CartoDB DarkMatter" pour dark mode

# ── Zones états australiens colorées par humidité ───────────────────
folium.Choropleth(
    geo_data=aus_geo,
    name="Zones climatiques",
    data=pred.groupby("ville")["prob_pluie"].mean().reset_index(),
    columns=["ville", "prob_pluie"],
    key_on="feature.properties.STATE_NAME",
    fill_color="Blues",
    fill_opacity=0.35,
    line_opacity=0.8,
    line_color="white",
    line_weight=2,
    legend_name="Probabilité de pluie moyenne",
    nan_fill_color="lightgray",
).add_to(m)

# ── Groupes par date (affichage / masquage via LayerControl) ─────────
DAY_FR   = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
MONTH_FR = ["","Jan","Fév","Mar","Avr","Mai","Jun",
            "Jul","Aoû","Sep","Oct","Nov","Déc"]

date_groups = {}
for date in dates:
    ts    = pd.Timestamp(date)
    label = f"{DAY_FR[ts.dayofweek]} {ts.day} {MONTH_FR[ts.month]}"
    group = folium.FeatureGroup(name=label, show=(date == dates[0]))
    df_day = pred[pred["date_predite"] == date]

    for _, row in df_day.iterrows():
        icon_html = weather_icon_html(
            row["prob_pluie"], row["pluie_prevue"], row["temp_predite"]
        )
        folium.Marker(
            location=[row["lat"], row["lon"]],
            popup=folium.Popup(popup_html(row), max_width=220),
            tooltip=f"<b>{row['ville']}</b> — {row['temp_predite']:.0f}°C",
            icon=folium.DivIcon(
                html=icon_html,
                icon_size=(60, 60),
                icon_anchor=(30, 55),
            ),
        ).add_to(group)

    group.add_to(m)
    date_groups[label] = group

# ── Contrôle des couches (sélecteur de dates) ────────────────────────
folium.LayerControl(
    position="topright",
    collapsed=False,
).add_to(m)

# ── Titre sur la carte ───────────────────────────────────────────────
title_html = """
<div style="
    position: fixed; top: 12px; left: 12px; z-index: 1000;
    background: white; padding: 10px 16px;
    border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.15);
    font-family: Arial, sans-serif;
">
    <b style="font-size: 16px; color: #1a3a5c">🌏 Météo Australie</b><br>
    <span style="font-size: 11px; color: #888">Prédictions par ville</span>
</div>"""
m.get_root().html.add_child(folium.Element(title_html))

m.save("australia_weather_map.html")
print("Carte exportée → australia_weather_map.html")

Carte exportée → australia_weather_map.html


In [39]:
# ── Supprimer le LayerControl ────────────────────────────────────────
# (ne pas ajouter folium.LayerControl)

# ── Injecter des boutons dates en JS ────────────────────────────────
import json

date_labels = []
for date in dates:
    ts = pd.Timestamp(date)
    date_labels.append(f"{DAY_FR[ts.dayofweek]}<br>{ts.day}<br>{MONTH_FR[ts.month].upper()}")

custom_js = f"""
<script>
(function() {{
    // Noms des couches dans l'ordre
    const dateLabels = {json.dumps(date_labels)};

    // Récupère toutes les FeatureGroups (couches de dates)
    const map = Object.values(window).find(v => v?._leaflet_id);

    // Attendre que la carte soit prête
    window.addEventListener('load', function() {{

        // ── Barre de boutons ────────────────────────────────────────
        const bar = document.createElement('div');
        bar.style.cssText = `
            position: fixed;
            top: 12px; left: 50%;
            transform: translateX(-50%);
            z-index: 1000;
            display: flex;
            gap: 4px;
            background: white;
            padding: 6px;
            border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.15);
            font-family: Arial, sans-serif;
        `;
        document.body.appendChild(bar);

        // Récupère les couches Leaflet depuis les checkboxes du LayerControl
        const checkboxes = document.querySelectorAll('.leaflet-control-layers-overlays input');

        checkboxes.forEach((cb, i) => {{
            const btn = document.createElement('button');
            btn.innerHTML = dateLabels[i] || `Jour ${{i+1}}`;
            btn.dataset.index = i;
            btn.style.cssText = `
                padding: 6px 14px;
                border: 1.5px solid #c0d4e4;
                border-radius: 7px;
                background: white;
                color: #1a3a5c;
                font-size: 12px;
                font-family: Arial, sans-serif;
                font-weight: bold;
                cursor: pointer;
                line-height: 1.4;
                text-align: center;
                transition: all 0.15s;
            `;

            // Activation au clic
            btn.addEventListener('click', function() {{
                // Désactiver toutes les couches
                checkboxes.forEach(c => {{ if (c.checked) c.click(); }});
                // Activer seulement celle-ci
                if (!cb.checked) cb.click();
                // Style actif/inactif
                bar.querySelectorAll('button').forEach(b => {{
                    b.style.background = 'white';
                    b.style.color = '#1a3a5c';
                    b.style.borderColor = '#c0d4e4';
                }});
                this.style.background = '#1a3a5c';
                this.style.color = 'white';
                this.style.borderColor = '#1a3a5c';
            }});

            bar.appendChild(btn);
        }});

        // Activer le premier bouton par défaut
        if (bar.children.length > 0) {{
            bar.children[0].click();
        }}

        // Cacher le LayerControl natif (plus utile)
        const layerControl = document.querySelector('.leaflet-control-layers');
        if (layerControl) layerControl.style.display = 'none';
    }});
}})();
</script>
"""

m.get_root().html.add_child(folium.Element(custom_js))
m.save("australia_weather_map_2.html")

# Après m.save(), injecte ce CSS pour masquer la ligne choroplèthe du LayerControl
hide_choropleth_css = """
<style>
/* Cache la 1ère ligne du LayerControl (la choroplèthe) */
.leaflet-control-layers-overlays label:first-child {
    display: none;
}
</style>
"""
m.get_root().html.add_child(folium.Element(hide_choropleth_css))